# Reproduce the MSR 2027 benchmark results

This notebook recalculates annotation statistics and prediction metrics from the released CSVs without acquiring repositories or calling an LLM. Each worksheet CSV contains seven columns and 101 rows. Unknown predictions are not negative labels. Independent scan-completeness flags define the heuristic evaluation cohorts. All 100 repositories informed heuristic development.


In [1]:
from pathlib import Path
import importlib.util
import json

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "results/benchmark-100/reproduce_paper.py").exists())
HERE = ROOT / "results/benchmark-100"
spec = importlib.util.spec_from_file_location("paper_reproduction", HERE / "reproduce_paper.py")
analysis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis)

# Set these to the original local files when available. Do not rename or edit manifests.
ORIGINAL_RUN = None  # e.g. ROOT / "data/runs/benchmark-100"
CORPUS_PREDICTIONS = None  # e.g. ROOT / "results/historical-386/predictions.csv"
report = analysis.analyze(ORIGINAL_RUN, CORPUS_PREDICTIONS)
print("Workbook SHA-256:", report["inputs"]["source_workbook_sha256"])
for item in report["inputs"]["sheets"]:
    print(item["role"], "<-", item["source_sheet"], item["range"], "101 rows x 7 columns")


Workbook SHA-256: 9331524372765d6c4898cdede32b2c9d0b654d4638d18c6f46358fcfbf9928a1
reviewer1 <- research_lifecycle_benchmark_10 A1:G101 101 rows x 7 columns
reviewer2 <- Hoja 10 A1:G101 101 rows x 7 columns
agreement <- agreement A1:G101 101 rows x 7 columns
final_agreement <- final_agreement A1:G101 101 rows x 7 columns
llm_documentation <- Hoja 11 A1:G101 101 rows x 7 columns
llm_code <- Hoja 12 A1:G101 101 rows x 7 columns
llm_combined <- Hoja 13 A1:G101 101 rows x 7 columns
heuristics_documentation <- benchmark_100_heuristics_doc_la A1:G101 101 rows x 7 columns
heuristics_code <- benchmark_100_heuristics_code_l A1:G101 101 rows x 7 columns
heuristics_combined <- benchmark_100_heuristics_both_l A1:G101 101 rows x 7 columns


## Human annotations and Table 3

Independent reviewer labels are compared by normalized repository identity, never by assumed row position. The agreement CSV is checked against all 600 independent comparisons. Final consensus is a separate reference.


In [2]:
annotations = report["annotations"]
claims = json.loads((HERE / "paper_claims.json").read_text())
for row in annotations["per_stage"]:
    assert row["agreement"] == claims["table_3"][row["stage"]]
    print(f"{row['stage']:18} agreement={row['agreement']:3} disagreement={row['disagreement']:3} final positive={row['final_positive']:3}")
print("Totals:", {key: value for key, value in annotations.items() if key != "per_stage"})
assert annotations["agreement"] == 354 and annotations["disagreement"] == 246
assert annotations["final_positive_labels"] == 384 and annotations["all_six_positive_repositories"] == 12


Collection         agreement= 53 disagreement= 47 final positive= 36
Processing         agreement= 55 disagreement= 45 final positive= 44
Method             agreement= 53 disagreement= 47 final positive= 71
Experimentation    agreement= 72 disagreement= 28 final positive= 88
Evaluation         agreement= 53 disagreement= 47 final positive= 66
Dissemination      agreement= 68 disagreement= 32 final positive= 79
Totals: {'agreement': 354, 'disagreement': 246, 'final_positive_labels': 384, 'final_negative_labels': 216, 'all_six_positive_repositories': 12}


## Metrics from the corrected exports

Missing (`?`, blank or Unverified) decisions are unknown, never zero. Heuristic evaluation additionally requires the original assessment scan-completeness flag. Known positives from incomplete scans remain in the CSVs. Metrics, confusion counts and baselines are calculated against final consensus.


In [3]:
print(f"{'Source':28} {'N':>3} {'P':>8} {'R':>8} {'Micro-F1':>9} {'Macro-F1':>9} {'Exact':>8}")
for role, result in report["workbook_metrics"].items():
    m = analysis.compact(result)
    print(f"{role:28} {m['repositories']:3} {m['micro_precision']:8.5f} {m['micro_recall']:8.5f} {m['micro_f1']:9.5f} {m['macro_f1']:9.5f} {m['exact_match']:8.5f}")
    print("  excluded:", result["excluded_repository_ids"])
    print("  always-positive micro-F1:", result["baselines"]["always_positive"]["micro"]["f1"])


Source                         N        P        R  Micro-F1  Macro-F1    Exact
llm_documentation             98  0.77855  0.88594   0.82878   0.80979  0.19388
  excluded: ['emergent-behaviors-in-biology/community-simulator', 'ihincks/nv-adaptive']
  always-positive micro-F1: 0.78134715
llm_code                      63  0.66292  0.97925   0.79062   0.76694  0.15873
  excluded: ['aquemy/dpso_experiments', 'aviranlab/sleq', 'b5strbal/macaw', 'christopherwilks/snaptron-experiments', 'ciaranm/cp2015-subgraph-isomorphism', 'cristal-smac/atom', 'diglesia/delaiglesiaetal_data', 'dmlb2000/eetup', 'drbenvincent/darc-experiments-matlab', 'drgriffis/sbd-evaluation', 'eglxiang/ytf', 'emergent-behaviors-in-biology/community-simulator', 'freedisciplina/collisionoffset', 'gmarceaucaron/natural-langevin-dynamics-for-neural-networks', 'ihincks/nv-adaptive', 'irit-melodi/irit-rst-dt', 'irit-melodi/irit-stac', 'jmschrei/pypore', 'joefresna/plosone-reina2015', 'liusi2019/ocd', 'md-k-sarker/ecii-expr', 'mi

## Same-cohort comparisons

The intersection is calculated separately for heuristic and LLM modes. Heuristic cohorts require original scan completeness; LLM cohorts require six binary labels. These comparisons do not establish independent code accuracy or held-out generalization.


In [4]:
for family, results in report["common_binary_complete_cohorts"].items():
    print(family)
    for role, result in results.items():
        print(role, "N=", result["repository_count"], "micro-F1=", result["micro"]["f1"], "macro-F1=", result["macro"]["f1"])


llm
llm_documentation N= 63 micro-F1= 0.84540117 macro-F1= 0.82007506
llm_code N= 63 micro-F1= 0.79061977 macro-F1= 0.76694479
llm_combined N= 63 micro-F1= 0.79338843 macro-F1= 0.76978086
heuristics
heuristics_documentation N= 95 micro-F1= 0.70569106 macro-F1= 0.62364393
heuristics_code N= 95 micro-F1= 0.41393443 macro-F1= 0.35307675
heuristics_combined N= 95 micro-F1= 0.75707899 macro-F1= 0.68340746


## Paper consistency checks

Compare calculated benchmark metrics with the reported paper values. Table 4 uses 96/95/95 eligible repositories; Table 6 uses 98/63/99. Combined LLM micro-F1 is reported as 0.795. The optional pooled 386 prediction export enables independent calculation of Table 5.


In [5]:
differences = [row for row in report["paper_comparison"] if row["status"] != "matches"]
for row in differences:
    print(row["table"], row["role"], row["metric"], "paper=", row["paper"], "computed=", row["computed"], row["status"])
assert all(row["status"] == "matches" for row in report["paper_comparison"] if row["table"] == "table_4")
print("Table 4: reproduced from original assessment predictions and scan-completeness flags")
print("Additional raw-run verification:", "recomputed" if report["original_run_metrics"] else "optional: supply ORIGINAL_RUN")
print("386-repository Table 5:", "recomputed" if report["corpus_stage_counts"] else "unavailable: supply CORPUS_PREDICTIONS")
if report["original_run_paper_comparison"]:
    print(json.dumps(report["original_run_paper_comparison"], indent=2))
if report["corpus_paper_comparison"]:
    print(json.dumps(report["corpus_paper_comparison"], indent=2))


Table 4: reproduced from original assessment predictions and scan-completeness flags
Additional raw-run verification: optional: supply ORIGINAL_RUN
386-repository Table 5: unavailable: supply CORPUS_PREDICTIONS


## Save benchmark exports and reproduce full-corpus counts

Save the benchmark calculations, then reproduce all 3,211-corpus stage aggregates from the preserved execution log and supplied 386-corpus stage counts. This calculation checks hashes and population membership. It does not recover missing original per-repository evidence or full-population accuracy.


In [6]:
analysis.write_outputs(report, HERE / "analysis")
print("Saved:", "results/benchmark-100/analysis")
for limitation in report["limitations"]:
    print("-", limitation)

corpus_spec = importlib.util.spec_from_file_location("corpus_reproduction", ROOT / "results/corpus-3211/reproduce_results.py")
corpus_analysis = importlib.util.module_from_spec(corpus_spec)
corpus_spec.loader.exec_module(corpus_analysis)
corpus_counts = corpus_analysis.reproduce()
corpus_analysis.write_outputs(corpus_counts)


Saved: results/benchmark-100/analysis
- Heuristic evaluation uses original assessment scan-completeness flags; known positives in incomplete scans are retained in CSVs.
- Paper Table 4 uses released predictions and assessment completeness flags (96/95/95 repositories).
- Paper Table 5 requires the original 386-repository predictions export.
- Table 7 summarizes the same predictions and confusion counts as the benchmark evaluation.
- Combined LLM micro-F1 is reported as 0.795 to three decimal places.
- All 100 repositories informed heuristic development; no held-out claim.
Full corpus: 3211 repositories. Combined counts: positive / negative / unknown
Collection          408 / 2219 /  584 (12.71% positive of all repositories)
Processing         1219 / 1479 /  513 (37.96% positive of all repositories)
Method             1324 / 1373 /  514 (41.23% positive of all repositories)
Experimentation    2007 /  738 /  466 (62.50% positive of all repositories)
Evaluation         1237 / 1453 /  521 